In [12]:
import pandas as pd 
import numpy as np
from xgboost import XGBClassifier
import matplotlib.pyplot as plt
import shap
import matplotlib; matplotlib.use('Agg')
from sklearn.model_selection import train_test_split, cross_val_score,StratifiedKFold
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import (classification_report, confusion_matrix,
ConfusionMatrixDisplay, roc_auc_score, roc_curve)
import joblib,os
os.makedirs('models',exist_ok=True)
df = pd.read_csv(r"C:\Users\niluc\Downloads\PROJECT\Churn\Data\churn_clean.csv")
X = df.drop('Churn',axis=1)
y = df['Churn']
X_train,X_test,y_train,y_test = train_test_split(
    X, y, test_size=0.2, random_state=42,stratify=y)
scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s = scaler.transform(X_test)
neg =(y_train == 0).sum()
pos =(y_train == 1).sum()
ratio = neg / pos
models = {
    'Random Forest' :       RandomForestClassifier(n_estimators=200, max_depth=10,
                                                   class_weight='balanced',random_state=42),
    'Gradient Boosting'   : GradientBoostingClassifier(n_estimators=150,
                                                       learning_rate=0.05, max_depth=4, random_state=42),
    'XGBoost'              :XGBClassifier(n_estimators = 400,scale_pos_weight=ratio, max_depth=4,learning_rate=0.01,min_child_weight = 5,gamma=0.2,
                                          subsample=0.8,random_state=42,colsample_bytree=0.8,reg_alpha=0.5,reg_lambda=1.0,n_jobs=-1,eval_metric='aucpr'
                        )}
results ={}
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
for name, m in models.items():
    m.fit(X_train_s, y_train)
    pred = m.predict(X_test_s)
    prob = m.predict_proba(X_test_s)[:,1]
    auc = roc_auc_score(y_test, prob)
    cv_s = cross_val_score(m, X_train_s, y_train, cv=cv, scoring= 'roc_auc')
    results[name] = {'auc' :auc,'prob' :prob,'model' :m, 'cv' :cv_s}
    print(f'\n=== {name} ===')
    print(classification_report(y_test,pred,target_names=['Stay','Churn']))
    print(f'AUC: {auc: .4f} | CV: {cv_s.mean():.4f} +/- {cv_s.std():.4f}')
    best_name = max(results, key=lambda k: results[k]['auc'])
    best_model = results[best_name]['model']
    print(f'\nBest: {best_name} AUC={results[best_name]["auc"]:.4f}')

    plt.figure(figsize=(7,5))
    for name, res in results.items():
        fpr,tpr,_=roc_curve(y_test, res['prob'])
        plt.plot(fpr, tpr, label=f'{name} ({res["auc"]:.3f})')
    plt.plot([0,1],[0,1],'k--');plt.legend()
    plt.title('ROC Curves');
    plt.tight_layout()
    plt.savefig('roc_curves.png', dpi=120)


    cm=confusion_matrix(y_test, best_model.predict(X_test_s))
    ConfusionMatrixDisplay(cm, display_labels=['Stay', 'Chun']).plot(cmap='Blues')
    plt.title(f'Confussion Matrix - {best_name}')
    plt.tight_layout();
    plt.savefig('confussion_matrix.png', dpi=120)
    plt.close()

joblib.dump(best_model,    'models/churn_modelv2.pk1')
joblib.dump(scaler,        'models/churn_scalerv2.pk1')
joblib.dump(list(X.columns),'models/churn_feature_namesv2.pk1')
print('Model saved.')

 
        
      

    
    
        
               
    


=== Random Forest ===
              precision    recall  f1-score   support

        Stay       0.89      0.79      0.84      1035
       Churn       0.56      0.72      0.63       374

    accuracy                           0.77      1409
   macro avg       0.72      0.76      0.73      1409
weighted avg       0.80      0.77      0.78      1409

AUC:  0.8430 | CV: 0.8433 +/- 0.0104

Best: Random Forest AUC=0.8430

=== Gradient Boosting ===
              precision    recall  f1-score   support

        Stay       0.84      0.90      0.87      1035
       Churn       0.66      0.52      0.58       374

    accuracy                           0.80      1409
   macro avg       0.75      0.71      0.73      1409
weighted avg       0.79      0.80      0.79      1409

AUC:  0.8451 | CV: 0.8451 +/- 0.0104

Best: Gradient Boosting AUC=0.8451

=== XGBoost ===
              precision    recall  f1-score   support

        Stay       0.91      0.73      0.81      1035
       Churn       0.52     

In [11]:
    explainer = shap.TreeExplainer(best_model)
    shap_vals = explainer.shap_values(X_test_s)
    sv = shap_vals[1] if isinstance(shap_vals,list) else shap_vals
    plt.figure(figsize=(10,8))
    shap.summary_plot(sv, pd.DataFrame(X_test_s, columns=X.columns),
                                       plot_type ='bar',show=False)
    plt.title('Churn top Drivers - SHAP')
    plt.tight_layout()
    plt.savefig('churn_shap_waterfall.png',dpi=150,bbox_inches ='tight')
    plt.close()
    print('SHAP Done!')

SHAP Done!
